In [ ]:
%load_ext autoreload
%autoreload 2

# Flight Delay ML Platform — Scheduled Traffic Models

## 1. Experimental Design

Compare the predictive performance of scheduled flight
features with and without scheduled departure traffic.

### Feature Sets

- V3-A: Scheduled baseline and route interactions
- V3-C1: V3-A plus two scheduled departure traffic features

### Evaluation Protocol

- Three expanding-window temporal CV folds within 2023
- Full training period: January–December 2023
- Development validation: January–August 2024
- Fixed previously optimized LightGBM hyperparameters
- Average Precision as the primary evaluation metric

Calibration, threshold-selection, and final test datasets
remain excluded.

Scheduled traffic features are retrospective schedule
proxies and not verified historical T-24h snapshots.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
)

from flight_delay.data.manifest import sha256_file
from flight_delay.data.storage import load_parquet

from flight_delay.features.build import (
    build_baseline_features,
)

from flight_delay.features.advanced_scheduled import (
    build_advanced_scheduled_features,
    INTERACTION_FEATURE_COLUMNS,
)

from flight_delay.features.scheduled_congestion import (
    DEPARTURE_TRAFFIC_FEATURES,
)

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").is_file()

PROCESSED_DIR = ROOT / "data/processed"
MANIFEST_DIR = ROOT / "data/interim/manifests"

TRAFFIC_DIR = (
    ROOT / "data/interim/scheduled_departure_traffic_v1"
)

with (TRAFFIC_DIR / "feature_manifest.json").open() as f:
    traffic_manifest = json.load(f)

with (
    ROOT / "artifacts/hyperparameter_optimization/best_params.json"
).open() as f:
    best_params = json.load(f)

IDENTITY_COLUMNS = traffic_manifest["identity_columns"]

BASELINE_INPUT_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]

CATEGORICAL_COLUMNS = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Airline_Route",
]

TARGETS = {
    "Departure": "DepDel15",
    "Arrival": "ArrDel15",
}

print("Feature version:", traffic_manifest["feature_version"])

display(pd.DataFrame(best_params))

print("\nEnvironment ready!")

In [ ]:
def load_congestion_dataset(start_month, end_month):

    parts = []
    targets = []
    dates = []

    months = pd.period_range(
        start_month, end_month, freq="M"
    ).astype(str)

    for month in months:

        entry = traffic_manifest["months"][month]

        with (MANIFEST_DIR / f"{month}.json").open() as f:
            manifest = json.load(f)

        raw_path = (
            PROCESSED_DIR
            / manifest["processed"]["filename"]
        )

        traffic_path = TRAFFIC_DIR / entry["filename"]

        assert sha256_file(raw_path) == (
            entry["processed_sha256"]
        )

        assert entry["processed_sha256"] == (
            manifest["processed"]["sha256"]
        )

        assert entry["source_sha256"] == (
            manifest["source"]["sha256"]
        )

        assert sha256_file(traffic_path) == entry["sha256"]

        raw = load_parquet(raw_path)
        traffic = pd.read_parquet(traffic_path)

        assert len(raw) == len(traffic) == entry["row_count"]

        # Exact flight identity and order verification.
        pd.testing.assert_frame_equal(
            raw[IDENTITY_COLUMNS].reset_index(drop=True),
            traffic[IDENTITY_COLUMNS],
            check_dtype=False,
        )

        baseline = build_baseline_features(
            raw[BASELINE_INPUT_COLUMNS]
        )

        advanced = build_advanced_scheduled_features(
            raw
        )[list(INTERACTION_FEATURE_COLUMNS)]

        congestion = traffic[
            list(DEPARTURE_TRAFFIC_FEATURES)
        ].copy()

        congestion.index = raw.index

        X = pd.concat(
            [baseline, advanced, congestion],
            axis=1,
        )

        assert X.shape == (len(raw), 23)
        assert not X.isna().any().any()
        assert len(set(X.columns)) == 23

        parts.append(X)

        targets.append(
            raw[["DepDel15", "ArrDel15"]]
        )

        dates.append(raw["FlightDate"])

        print(f"{month} | records={len(X):,}")

        del raw, traffic, baseline, advanced, congestion

    return {
        "X": pd.concat(parts, ignore_index=True),
        "y": pd.concat(targets, ignore_index=True),
        "flight_dates": pd.concat(dates, ignore_index=True),
    }


train = load_congestion_dataset(
    "2023-01", "2023-12"
)

validation = load_congestion_dataset(
    "2024-01", "2024-08"
)

assert train["X"].shape == (1_478_412, 23)
assert validation["X"].shape == (1_020_082, 23)

assert train["X"].columns.equals(
    validation["X"].columns
)

FEATURE_SETS = {
    "V3-A": [
        col for col in train["X"].columns
        if col not in DEPARTURE_TRAFFIC_FEATURES
    ],
    "V3-C1": train["X"].columns.tolist(),
}

assert len(FEATURE_SETS["V3-A"]) == 21
assert len(FEATURE_SETS["V3-C1"]) == 23

print("\n=== Model Dataset Summary ===")

print("Training:", train["X"].shape)
print("Validation:", validation["X"].shape)

print("\nModel dataset loading passed!")

In [ ]:
def prepare_lgb_features(
    train_frame,
    validation_frame,
    feature_set,
):

    columns = FEATURE_SETS[feature_set]

    X_tr = train_frame[columns].copy()
    X_val = validation_frame[columns].copy()

    for column in CATEGORICAL_COLUMNS:

        categories = sorted(
            X_tr[column].astype("string").unique().tolist()
        )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        for frame in [X_tr, X_val]:

            values = frame[column].astype("string")

            values = values.where(
                values.isin(categories),
                pd.NA,
            )

            frame[column] = values.astype(dtype)

        assert X_tr[column].notna().all()

        assert X_tr[column].dtype == X_val[column].dtype

    numeric_columns = [
        col for col in columns
        if col not in CATEGORICAL_COLUMNS
    ]

    for frame in [X_tr, X_val]:

        assert np.isfinite(
            frame[numeric_columns].to_numpy(dtype=float)
        ).all()

    return X_tr, X_val

In [ ]:
train_months = (
    pd.to_datetime(train["flight_dates"])
    .dt.to_period("M")
    .astype(str)
)

CV_FOLDS = [
    ("2023-06", "2023-07", "2023-08"),
    ("2023-08", "2023-09", "2023-10"),
    ("2023-10", "2023-11", "2023-12"),
]

cv_records = []

for fold, (train_end, val_start, val_end) in enumerate(
    CV_FOLDS, start=1
):

    tr_idx = np.flatnonzero(
        train_months.le(train_end).to_numpy()
    )

    va_idx = np.flatnonzero(
        train_months.between(
            val_start, val_end
        ).to_numpy()
    )

    assert not np.intersect1d(tr_idx, va_idx).size

    for feature_set in FEATURE_SETS:

        X_tr, X_val = prepare_lgb_features(
            train["X"].iloc[tr_idx],
            train["X"].iloc[va_idx],
            feature_set,
        )

        for task, target in TARGETS.items():

            y_tr = train["y"][target].iloc[tr_idx]
            y_val = train["y"][target].iloc[va_idx]

            assert X_tr.index.equals(y_tr.index)
            assert X_val.index.equals(y_val.index)

            model = LGBMClassifier(
                **best_params[task]
            )

            model.fit(
                X_tr,
                y_tr,
                categorical_feature=CATEGORICAL_COLUMNS,
            )

            probabilities = model.predict_proba(
                X_val
            )[:, 1]

            ap = average_precision_score(
                y_val, probabilities
            )

            cv_records.append({
                "Fold": fold,
                "FeatureSet": feature_set,
                "Task": task,
                "AP": ap,
            })

            print(
                f"Fold {fold} | {feature_set} | "
                f"{task} | AP={ap:.6f}"
            )

            del model, probabilities

        del X_tr, X_val

congestion_cv = pd.DataFrame(cv_records)

assert len(congestion_cv) == 12

congestion_cv_comparison = (
    congestion_cv.groupby(
        ["Task", "FeatureSet"]
    )["AP"]
    .mean()
    .unstack()
)

congestion_cv_comparison["TrafficGain"] = (
    congestion_cv_comparison["V3-C1"]
    - congestion_cv_comparison["V3-A"]
)

display(congestion_cv_comparison.round(6))

In [ ]:
def evaluate_model(y_true, probabilities):

    probabilities = np.asarray(probabilities)

    assert np.isfinite(probabilities).all()
    assert len(y_true) == len(probabilities)

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "AP": average_precision_score(
            y_true, probabilities
        ),
        "ROC_AUC": roc_auc_score(
            y_true, probabilities
        ),
        "Accuracy": accuracy_score(
            y_true, predictions
        ),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "Brier": brier_score_loss(
            y_true, probabilities
        ),
    }

In [ ]:
congestion_models = {}
congestion_probabilities = {}
evaluation_records = []

for feature_set in FEATURE_SETS:

    X_tr, X_val = prepare_lgb_features(
        train["X"],
        validation["X"],
        feature_set,
    )

    for task, target in TARGETS.items():

        y_tr = train["y"][target]
        y_val = validation["y"][target]

        assert X_tr.index.equals(y_tr.index)
        assert X_val.index.equals(y_val.index)

        model = LGBMClassifier(
            **best_params[task]
        )

        start = time.perf_counter()

        model.fit(
            X_tr,
            y_tr,
            categorical_feature=CATEGORICAL_COLUMNS,
        )

        train_seconds = time.perf_counter() - start

        assert model.classes_.tolist() == [0, 1]

        start = time.perf_counter()

        probabilities = model.predict_proba(
            X_val
        )[:, 1]

        inference_seconds = time.perf_counter() - start

        train_prob = model.predict_proba(
            X_tr
        )[:, 1]

        train_ap = average_precision_score(
            y_tr, train_prob
        )

        metrics = evaluate_model(
            y_val, probabilities
        )

        result = {
            "Model": "LightGBM Tuned",
            "Task": task,
            "FeatureSet": feature_set,
            "FeatureCount": X_tr.shape[1],
            "TrainingSeconds": train_seconds,
            "InferenceSeconds": inference_seconds,
            "TrainAP": train_ap,
            "GeneralizationGap": train_ap - metrics["AP"],
            **metrics,
        }

        evaluation_records.append(result)

        congestion_models[
            (feature_set, task)
        ] = model

        congestion_probabilities[
            (feature_set, task)
        ] = probabilities

        display(pd.DataFrame([result]).round(6))

        del train_prob

    del X_tr, X_val

congestion_evaluation = pd.DataFrame(
    evaluation_records
)

assert len(congestion_evaluation) == 4

print("\nScheduled traffic model training completed!")

In [ ]:
congestion_comparison = (
    congestion_evaluation.pivot(
        index="Task",
        columns="FeatureSet",
        values="AP",
    )[["V3-A", "V3-C1"]]
)

congestion_comparison["AbsoluteGain"] = (
    congestion_comparison["V3-C1"]
    - congestion_comparison["V3-A"]
)

congestion_comparison["RelativeGainPct"] = (
    congestion_comparison["AbsoluteGain"]
    / congestion_comparison["V3-A"]
    * 100
)

print("=== Scheduled Traffic Ablation ===")

display(congestion_evaluation.round(6))

display(congestion_comparison.round(6))

ax = congestion_comparison[
    ["V3-A", "V3-C1"]
].plot.bar(
    figsize=(9, 5),
    rot=0,
)

ax.set_title(
    "Scheduled Departure Traffic Ablation"
)

ax.set_ylabel("Validation Average Precision")
ax.set_ylim(0, max(
    0.5,
    congestion_comparison[["V3-A", "V3-C1"]]
    .max().max() * 1.15,
))

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.tight_layout()
plt.show()

In [ ]:
RESULT_DIR = ROOT / "artifacts/scheduled_congestion"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "cv_results": congestion_cv,
    "cv_comparison": congestion_cv_comparison.reset_index(),
    "validation_results": congestion_evaluation,
    "ap_comparison": congestion_comparison.reset_index(),
}

for name, table in tables.items():

    table.to_csv(
        RESULT_DIR / f"{name}.csv",
        index=False,
    )

print("Scheduled congestion experiment results saved!")

## 7. CatBoost Model Comparison

### Objective

Investigate whether CatBoost improves flight delay
prediction through alternative categorical feature
processing and boosting mechanisms.

### Experimental Protocol

- Feature set: V3-A (21 scheduled features)
- Training period: January–December 2023
- Three expanding-window temporal CV folds
- Development validation: January–August 2024
- Primary metric: Average Precision
- Random seed: 42

The experiment uses a fixed initial CatBoost
configuration without additional hyperparameter tuning.

Categorical statistics are learned exclusively from
each training partition.

The calibration, threshold-selection, and final test
datasets remain excluded.

Results are compared against previously optimized
LightGBM models using identical feature sets and cohorts.

In [ ]:
from catboost import CatBoostClassifier

CATBOOST_FEATURES = FEATURE_SETS["V3-A"]

CATBOOST_CATEGORICAL_COLUMNS = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Airline_Route",
]


def prepare_catboost_features(frame):

    X = frame[CATBOOST_FEATURES].copy()

    for column in CATBOOST_CATEGORICAL_COLUMNS:

        if X[column].isna().any():
            raise ValueError(
                f"Missing categorical values: {column}"
            )

        X[column] = (
            X[column].astype(str).astype(object)
        )

    numeric_columns = [
        col for col in CATBOOST_FEATURES
        if col not in CATBOOST_CATEGORICAL_COLUMNS
    ]

    assert np.isfinite(
        X[numeric_columns].to_numpy(dtype=float)
    ).all()

    assert X.index.equals(frame.index)

    return X


X_train_cat = prepare_catboost_features(
    train["X"]
)

X_validation_cat = prepare_catboost_features(
    validation["X"]
)

assert X_train_cat.shape == (1_478_412, 21)
assert X_validation_cat.shape == (1_020_082, 21)

print("Training:", X_train_cat.shape)
print("Validation:", X_validation_cat.shape)

display(X_train_cat.dtypes)

print("\nCatBoost feature preparation passed!")

In [ ]:
CATBOOST_PARAMS = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "random_seed": 42,
    "thread_count": 4,
    "task_type": "CPU",
    "verbose": False,
    "allow_writing_files": False,
}

display(
    pd.Series(
        CATBOOST_PARAMS,
        name="Value",
    )
)

In [ ]:
catboost_cv_records = []

for fold, (train_end, val_start, val_end) in enumerate(
    CV_FOLDS,
    start=1,
):

    tr_idx = np.flatnonzero(
        train_months.le(train_end).to_numpy()
    )

    va_idx = np.flatnonzero(
        train_months.between(
            val_start, val_end
        ).to_numpy()
    )

    assert not np.intersect1d(tr_idx, va_idx).size

    assert (
        train_months.iloc[tr_idx].max()
        < train_months.iloc[va_idx].min()
    )

    X_tr = X_train_cat.iloc[tr_idx]
    X_val = X_train_cat.iloc[va_idx]

    for task, target in TARGETS.items():

        y_tr = train["y"][target].iloc[tr_idx]
        y_val = train["y"][target].iloc[va_idx]

        assert X_tr.index.equals(y_tr.index)
        assert X_val.index.equals(y_val.index)

        model = CatBoostClassifier(
            **CATBOOST_PARAMS
        )

        start = time.perf_counter()

        model.fit(
            X_tr,
            y_tr,
            cat_features=CATBOOST_CATEGORICAL_COLUMNS,
        )

        train_seconds = time.perf_counter() - start

        probabilities = model.predict_proba(
            X_val
        )[:, 1]

        ap = average_precision_score(
            y_val,
            probabilities,
        )

        catboost_cv_records.append({
            "Model": "CatBoost",
            "FeatureSet": "V3-A",
            "Task": task,
            "Fold": fold,
            "AP": ap,
            "TrainingSeconds": train_seconds,
        })

        print(
            f"Fold {fold} | "
            f"{task} | "
            f"AP={ap:.6f} | "
            f"Training={train_seconds:.2f}s"
        )

        del model, probabilities

    del X_tr, X_val

catboost_cv = pd.DataFrame(
    catboost_cv_records
)

assert len(catboost_cv) == 6

print("\nCatBoost temporal CV completed!")

In [ ]:
lgb_cv_reference = congestion_cv.loc[
    congestion_cv["FeatureSet"].eq("V3-A")
].copy()

lgb_cv_reference["Model"] = "LightGBM Tuned"

model_cv_comparison = pd.concat(
    [
        lgb_cv_reference,
        catboost_cv,
    ],
    ignore_index=True,
)

model_cv_summary = (
    model_cv_comparison
    .groupby(["Task", "Model"])
    .agg(
        MeanAP=("AP", "mean"),
        StdAP=("AP", "std"),
        MinAP=("AP", "min"),
        MaxAP=("AP", "max"),
    )
    .reset_index()
)

display(model_cv_summary.round(6))

catboost_cv_comparison = (
    model_cv_summary.pivot(
        index="Task",
        columns="Model",
        values="MeanAP",
    )
)

catboost_cv_comparison["CatBoostGain"] = (
    catboost_cv_comparison["CatBoost"]
    - catboost_cv_comparison["LightGBM Tuned"]
)

display(catboost_cv_comparison.round(6))

In [ ]:
catboost_models = {}
catboost_probabilities = {}
catboost_results = []

for task, target in TARGETS.items():

    print(f"\n=== CatBoost V3-A: {task} ===")

    y_tr = train["y"][target]
    y_val = validation["y"][target]

    assert X_train_cat.index.equals(y_tr.index)
    assert X_validation_cat.index.equals(y_val.index)

    model = CatBoostClassifier(
        **CATBOOST_PARAMS
    )

    start = time.perf_counter()

    model.fit(
        X_train_cat,
        y_tr,
        cat_features=CATBOOST_CATEGORICAL_COLUMNS,
    )

    training_seconds = time.perf_counter() - start

    assert model.classes_.tolist() == [0, 1]

    start = time.perf_counter()

    val_prob = model.predict_proba(
        X_validation_cat
    )[:, 1]

    inference_seconds = time.perf_counter() - start

    train_prob = model.predict_proba(
        X_train_cat
    )[:, 1]

    train_ap = average_precision_score(
        y_tr,
        train_prob,
    )

    metrics = evaluate_model(
        y_val,
        val_prob,
    )

    result = {
        "Model": "CatBoost",
        "Task": task,
        "FeatureSet": "V3-A",
        "FeatureCount": len(CATBOOST_FEATURES),
        "TrainingSeconds": training_seconds,
        "InferenceSeconds": inference_seconds,
        "Trees": model.tree_count_,
        "TrainAP": train_ap,
        "GeneralizationGap": (
            train_ap - metrics["AP"]
        ),
        **metrics,
    }

    catboost_models[task] = model

    catboost_probabilities[task] = val_prob

    catboost_results.append(result)

    display(
        pd.DataFrame([result]).round(6)
    )

    del train_prob

catboost_evaluation = pd.DataFrame(
    catboost_results
)

print("\nCatBoost evaluation completed!")

In [ ]:
catboost_comparison = pd.concat(
    [
        congestion_evaluation,
        catboost_evaluation,
    ],
    ignore_index=True,
)

comparison_columns = [
    "Task",
    "Model",
    "FeatureSet",
    "AP",
    "ROC_AUC",
    "Precision",
    "Recall",
    "F1",
    "Brier",
    "TrainAP",
    "GeneralizationGap",
    "TrainingSeconds",
]

display(
    catboost_comparison[comparison_columns]
    .sort_values(
        ["Task", "AP"],
        ascending=[True, False],
    )
    .round(6)
)

In [ ]:
plot_data = catboost_comparison.copy()

plot_data["Configuration"] = (
    plot_data["Model"]
    + " "
    + plot_data["FeatureSet"]
)

plot_data = plot_data.pivot(
    index="Task",
    columns="Configuration",
    values="AP",
)

ax = plot_data.plot.bar(
    figsize=(10, 5),
    rot=0,
)

ax.set_title(
    "LightGBM vs CatBoost: Validation AP"
)

ax.set_ylabel("Average Precision")

ax.set_ylim(
    0,
    max(0.5, plot_data.max().max() * 1.15)
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.tight_layout()
plt.show()

In [ ]:
CATBOOST_DIR = ROOT / "artifacts/catboost_comparison"

CATBOOST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tables = {
    "cv_results": catboost_cv,
    "cv_comparison": catboost_cv_comparison.reset_index(),
    "validation_results": catboost_evaluation,
    "model_comparison": catboost_comparison,
}

for name, table in tables.items():

    table.to_csv(
        CATBOOST_DIR / f"{name}.csv",
        index=False,
    )

with (CATBOOST_DIR / "params.json").open(
    "w", encoding="utf-8"
) as f:

    json.dump(
        CATBOOST_PARAMS,
        f,
        indent=2,
    )

print("CatBoost experiment results saved!")

## 8. Seasonally Matched Historical Feature Ablation

### Objective

Evaluate whether previous-year same-month historical
delay rates improve individual-flight delay prediction.

### Feature Configurations

- V3-A: Scheduled features and route interactions
- V4-A: V3-A plus two seasonal origin delay rates
- V4-B: V4-A plus two seasonal airline-origin delay rates

### Controlled Variables

- Identical training and validation cohorts
- Fixed optimized LightGBM hyperparameters
- Identical temporal cross-validation folds
- Training-only categorical encoding
- Training-only missing-value imputation

Average Precision remains the primary evaluation metric.

Calibration, threshold-selection, and final test
datasets remain excluded.

In [ ]:
from flight_delay.features.seasonal_history import (
    SEASONAL_FEATURE_COLUMNS,
)

SEASONAL_DIR = ROOT / "data/interim/seasonal_history_v1"

with (
    SEASONAL_DIR / "feature_manifest.json"
).open() as f:

    seasonal_manifest = json.load(f)

assert seasonal_manifest["feature_version"] == (
    "seasonal_history_v1"
)

assert seasonal_manifest["feature_columns"] == list(
    SEASONAL_FEATURE_COLUMNS
)

assert seasonal_manifest["identity_columns"] == IDENTITY_COLUMNS


def load_seasonal_features(start_month, end_month):

    parts = []
    dates = []

    months = pd.period_range(
        start_month,
        end_month,
        freq="M",
    ).astype(str)

    for month in months:

        entry = seasonal_manifest["months"][month]

        source_month = str(
            pd.Period(month, freq="M") - 12
        )

        assert entry["source_month"] == source_month

        path = SEASONAL_DIR / entry["filename"]

        assert sha256_file(path) == entry["sha256"]

        with (MANIFEST_DIR / f"{month}.json").open() as f:
            target_manifest = json.load(f)

        with (
            MANIFEST_DIR / f"{source_month}.json"
        ).open() as f:
            source_manifest = json.load(f)

        target_path = (
            PROCESSED_DIR
            / target_manifest["processed"]["filename"]
        )

        source_path = (
            PROCESSED_DIR
            / source_manifest["processed"]["filename"]
        )

        assert sha256_file(target_path) == (
            entry["target_processed_sha256"]
        )

        assert sha256_file(source_path) == (
            entry["source_processed_sha256"]
        )

        target = load_parquet(target_path)
        historical = pd.read_parquet(path)

        assert len(target) == len(historical)
        assert len(historical) == entry["row_count"]

        pd.testing.assert_frame_equal(
            target[IDENTITY_COLUMNS].reset_index(drop=True),
            historical[IDENTITY_COLUMNS].reset_index(drop=True),
            check_dtype=False,
        )

        parts.append(
            historical[list(SEASONAL_FEATURE_COLUMNS)]
        )

        dates.append(target["FlightDate"])

        print(f"{month} | records={len(historical):,}")

        del target, historical

    return (
        pd.concat(parts, ignore_index=True),
        pd.concat(dates, ignore_index=True),
    )


train_seasonal, train_seasonal_dates = (
    load_seasonal_features("2023-01", "2023-12")
)

validation_seasonal, validation_seasonal_dates = (
    load_seasonal_features("2024-01", "2024-08")
)

pd.testing.assert_series_equal(
    train_seasonal_dates,
    train["flight_dates"],
    check_dtype=False,
    check_names=False,
)

pd.testing.assert_series_equal(
    validation_seasonal_dates,
    validation["flight_dates"],
    check_dtype=False,
    check_names=False,
)

print("\nTraining seasonal features:", train_seasonal.shape)

print(
    "Validation seasonal features:",
    validation_seasonal.shape,
)

print("\nSeasonal feature loading passed!")

In [ ]:
V3A_COLUMNS = FEATURE_SETS["V3-A"].copy()

X_train_v4 = pd.concat(
    [
        train["X"][V3A_COLUMNS],
        train_seasonal,
    ],
    axis=1,
)

X_validation_v4 = pd.concat(
    [
        validation["X"][V3A_COLUMNS],
        validation_seasonal,
    ],
    axis=1,
)

V4_FEATURE_SETS = {
    "V3-A": V3A_COLUMNS,
    "V4-A": (
        V3A_COLUMNS
        + list(SEASONAL_FEATURE_COLUMNS[:2])
    ),
    "V4-B": (
        V3A_COLUMNS
        + list(SEASONAL_FEATURE_COLUMNS)
    ),
}

assert X_train_v4.shape == (1_478_412, 25)

assert X_validation_v4.shape == (1_020_082, 25)

assert X_train_v4.index.equals(train["y"].index)

assert X_validation_v4.index.equals(
    validation["y"].index
)

assert X_train_v4.columns.is_unique

for name, columns in V4_FEATURE_SETS.items():

    print(f"{name}: {len(columns)} features")

print("\n=== Missing Seasonal Features ===")

display(pd.DataFrame({
    "Training": train_seasonal.isna().sum(),
    "Validation": validation_seasonal.isna().sum(),
}))

assert train_seasonal.isna().any(axis=1).sum() == 2

assert not validation_seasonal.isna().any().any()

print("\nV4 feature sets validated!")

In [ ]:
def prepare_seasonal_lgb_features(
    train_frame,
    validation_frame,
    feature_set,
):

    columns = V4_FEATURE_SETS[feature_set]

    X_tr = train_frame[columns].copy()
    X_val = validation_frame[columns].copy()

    # Training-only category levels.
    for column in CATEGORICAL_COLUMNS:

        categories = sorted(
            X_tr[column]
            .astype("string")
            .unique()
            .tolist()
        )

        dtype = pd.CategoricalDtype(
            categories=categories
        )

        for frame in [X_tr, X_val]:

            values = frame[column].astype("string")

            values = values.where(
                values.isin(categories),
                pd.NA,
            )

            frame[column] = values.astype(dtype)

        assert X_tr[column].notna().all()

        assert (
            X_tr[column].dtype
            == X_val[column].dtype
        )

    seasonal_columns = [
        col for col in SEASONAL_FEATURE_COLUMNS
        if col in columns
    ]

    # Fit imputation values exclusively on training.
    if seasonal_columns:

        medians = X_tr[seasonal_columns].median()

        assert medians.notna().all()

        X_tr[seasonal_columns] = (
            X_tr[seasonal_columns].fillna(medians)
        )

        X_val[seasonal_columns] = (
            X_val[seasonal_columns].fillna(medians)
        )

    numeric_columns = [
        col for col in columns
        if col not in CATEGORICAL_COLUMNS
    ]

    for frame in [X_tr, X_val]:

        assert np.isfinite(
            frame[numeric_columns].to_numpy(dtype=float)
        ).all()

    assert X_tr.index.equals(train_frame.index)

    assert X_val.index.equals(validation_frame.index)

    return X_tr, X_val

In [ ]:
train_months = (
    pd.to_datetime(train["flight_dates"])
    .dt.to_period("M")
    .astype(str)
)

CV_FOLDS = [
    ("2023-06", "2023-07", "2023-08"),
    ("2023-08", "2023-09", "2023-10"),
    ("2023-10", "2023-11", "2023-12"),
]

seasonal_cv_records = []

for fold, (train_end, val_start, val_end) in enumerate(
    CV_FOLDS,
    start=1,
):

    tr_idx = np.flatnonzero(
        train_months.le(train_end).to_numpy()
    )

    va_idx = np.flatnonzero(
        train_months.between(
            val_start, val_end
        ).to_numpy()
    )

    assert not np.intersect1d(tr_idx, va_idx).size

    assert (
        train_months.iloc[tr_idx].max()
        < train_months.iloc[va_idx].min()
    )

    for feature_set in V4_FEATURE_SETS:

        X_tr, X_val = prepare_seasonal_lgb_features(
            X_train_v4.iloc[tr_idx],
            X_train_v4.iloc[va_idx],
            feature_set,
        )

        for task, target in TARGETS.items():

            y_tr = train["y"][target].iloc[tr_idx]
            y_val = train["y"][target].iloc[va_idx]

            assert X_tr.index.equals(y_tr.index)
            assert X_val.index.equals(y_val.index)

            model = LGBMClassifier(
                **best_params[task]
            )

            start = time.perf_counter()

            model.fit(
                X_tr,
                y_tr,
                categorical_feature=CATEGORICAL_COLUMNS,
            )

            train_seconds = time.perf_counter() - start

            probabilities = model.predict_proba(
                X_val
            )[:, 1]

            ap = average_precision_score(
                y_val,
                probabilities,
            )

            seasonal_cv_records.append({
                "Fold": fold,
                "FeatureSet": feature_set,
                "Task": task,
                "AP": ap,
                "TrainingSeconds": train_seconds,
            })

            print(
                f"Fold {fold} | "
                f"{feature_set} | "
                f"{task} | AP={ap:.6f}"
            )

            del model, probabilities

        del X_tr, X_val

seasonal_cv = pd.DataFrame(seasonal_cv_records)

assert len(seasonal_cv) == 18

assert not seasonal_cv.duplicated(
    ["Fold", "FeatureSet", "Task"]
).any()

print("\nSeasonal temporal CV completed!")

In [ ]:
seasonal_cv_summary = (
    seasonal_cv.groupby(["Task", "FeatureSet"])
    .agg(
        MeanAP=("AP", "mean"),
        StdAP=("AP", "std"),
        MinAP=("AP", "min"),
        MaxAP=("AP", "max"),
    )
    .reset_index()
)

display(seasonal_cv_summary.round(6))

seasonal_cv_comparison = (
    seasonal_cv_summary.pivot(
        index="Task",
        columns="FeatureSet",
        values="MeanAP",
    )[["V3-A", "V4-A", "V4-B"]]
)

seasonal_cv_comparison["OriginGain"] = (
    seasonal_cv_comparison["V4-A"]
    - seasonal_cv_comparison["V3-A"]
)

seasonal_cv_comparison["AirlineGain"] = (
    seasonal_cv_comparison["V4-B"]
    - seasonal_cv_comparison["V4-A"]
)

seasonal_cv_comparison["TotalGain"] = (
    seasonal_cv_comparison["V4-B"]
    - seasonal_cv_comparison["V3-A"]
)

display(seasonal_cv_comparison.round(6))

In [ ]:
seasonal_models = {}
seasonal_probabilities = {}
seasonal_results = []

for feature_set in V4_FEATURE_SETS:

    X_tr, X_val = prepare_seasonal_lgb_features(
        X_train_v4,
        X_validation_v4,
        feature_set,
    )

    for task, target in TARGETS.items():

        print(f"\n=== {feature_set}: {task} ===")

        y_tr = train["y"][target]
        y_val = validation["y"][target]

        model = LGBMClassifier(
            **best_params[task]
        )

        start = time.perf_counter()

        model.fit(
            X_tr,
            y_tr,
            categorical_feature=CATEGORICAL_COLUMNS,
        )

        training_seconds = (
            time.perf_counter() - start
        )

        assert model.classes_.tolist() == [0, 1]

        start = time.perf_counter()

        val_prob = model.predict_proba(X_val)[:, 1]

        inference_seconds = (
            time.perf_counter() - start
        )

        train_prob = model.predict_proba(X_tr)[:, 1]

        train_ap = average_precision_score(
            y_tr,
            train_prob,
        )

        metrics = evaluate_model(
            y_val,
            val_prob,
        )

        result = {
            "Model": "LightGBM Tuned",
            "Task": task,
            "FeatureSet": feature_set,
            "FeatureCount": X_tr.shape[1],
            "TrainingSeconds": training_seconds,
            "InferenceSeconds": inference_seconds,
            "TrainAP": train_ap,
            "GeneralizationGap": (
                train_ap - metrics["AP"]
            ),
            **metrics,
        }

        seasonal_models[
            (feature_set, task)
        ] = model

        seasonal_probabilities[
            (feature_set, task)
        ] = val_prob

        seasonal_results.append(result)

        display(pd.DataFrame([result]).round(6))

        del train_prob

    del X_tr, X_val

seasonal_evaluation = pd.DataFrame(
    seasonal_results
)

assert len(seasonal_evaluation) == 6

print("\nSeasonal model training completed!")

In [ ]:
seasonal_comparison = (
    seasonal_evaluation.pivot(
        index="Task",
        columns="FeatureSet",
        values="AP",
    )[["V3-A", "V4-A", "V4-B"]]
)

seasonal_comparison["OriginGain"] = (
    seasonal_comparison["V4-A"]
    - seasonal_comparison["V3-A"]
)

seasonal_comparison["AirlineGain"] = (
    seasonal_comparison["V4-B"]
    - seasonal_comparison["V4-A"]
)

seasonal_comparison["TotalGain"] = (
    seasonal_comparison["V4-B"]
    - seasonal_comparison["V3-A"]
)

seasonal_comparison["RelativeGainPct"] = (
    seasonal_comparison["TotalGain"]
    / seasonal_comparison["V3-A"]
    * 100
)

print("=== Seasonal Ablation Results ===")

display(seasonal_evaluation.round(6))

print("\n=== Incremental Seasonal Feature Value ===")

display(seasonal_comparison.round(6))

In [ ]:
ax = seasonal_comparison[
    ["V3-A", "V4-A", "V4-B"]
].plot.bar(
    figsize=(9, 5),
    rot=0,
)

ax.set_title(
    "Seasonally Matched Historical Feature Ablation"
)

ax.set_ylabel("Validation Average Precision")

ax.set_ylim(
    0,
    max(
        0.5,
        seasonal_comparison[
            ["V3-A", "V4-A", "V4-B"]
        ].max().max() * 1.15,
    ),
)

for container in ax.containers:

    ax.bar_label(
        container,
        fmt="%.4f",
        padding=3,
    )

plt.tight_layout()
plt.show()

In [ ]:
RESULT_DIR = ROOT / "artifacts/seasonal_models"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "cv_results": seasonal_cv,
    "cv_summary": seasonal_cv_summary,
    "cv_comparison": seasonal_cv_comparison.reset_index(),
    "validation_results": seasonal_evaluation,
    "ap_comparison": seasonal_comparison.reset_index(),
}

for name, table in tables.items():

    table.to_csv(
        RESULT_DIR / f"{name}.csv",
        index=False,
    )

with (RESULT_DIR / "experiment_config.json").open(
    "w", encoding="utf-8"
) as f:

    json.dump(
        {
            "feature_sets": V4_FEATURE_SETS,
            "model_parameters": best_params,
            "cv_folds": CV_FOLDS,
            "availability_policy": "research_proxy",
            "imputation": "training_only_median",
        },
        f,
        indent=2,
    )

print("Seasonal model experiment results saved!")